# Generate 10k Gray-Scott snapshots

This notebook clones the dataset generator repo, simulates Gray-Scott trajectories after burn-in, saves snapshots in chunks of 1000 images, and downloads every chunk to your laptop.

In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Public repo works as is. For a private repo, add a Colab Secret named GITHUB_TOKEN.
import subprocess
from google.colab import userdata

REPO_URL = "https://github.com/SadreevAmir/DL_final_project.git"
REPO_DIR = "gray_scott_dataset_repo"

try:
    github_token = userdata.get("GITHUB_TOKEN")
except Exception:
    github_token = None

clone_cmd = ["git", "clone", REPO_URL, REPO_DIR]
if github_token:
    clone_cmd = [
        "git",
        "-c",
        f"http.extraHeader=Authorization: Bearer {github_token}",
        "clone",
        REPO_URL,
        REPO_DIR,
    ]

subprocess.run(["rm", "-rf", REPO_DIR], check=True)
subprocess.run(clone_cmd, check=True)
%cd {REPO_DIR}
!pip install -q -r requirements.txt

In [ ]:
from pathlib import Path
from google.colab import files
from IPython.display import display
from PIL import Image

from grayscott_dataset import GrayScottConfig, generate_dataset

output_dir = Path("data/grayscott_64")

config = GrayScottConfig(
    output_dir=str(output_dir),
    total_images=10_000,
    grid_size=64,
    num_trajectories=500,
    max_trajectories=2_000,
    snapshots_per_trajectory=20,
    burn_in_steps=2_500,
    save_interval=30,
    chunk_size=1_000,
    sim_batch_size=500,
    channels="v",
    dtype="float16",
    compress=False,
    param_mode="mixed",
    seed=42,
    device="cuda" if torch.cuda.is_available() else "cpu",
    num_threads=12,
    save_previews=True,
    preview_every_chunks=1,
    preview_count=32,
    min_image_std=0.025,
    min_image_range=0.15,
)

def download_chunk(path: Path):
    preview = path.with_name(path.name.replace("grayscott_chunk_", "preview_chunk_").replace(".npz", ".png"))
    if preview.exists():
        display(Image.open(preview))
        files.download(str(preview))
    print(f"Downloading {path} ...")
    files.download(str(path))

paths = generate_dataset(config, on_chunk_saved=download_chunk)
print(f"Saved {len(paths)} chunks")
files.download(str(output_dir / "manifest.json"))

In [ ]:
# Optional visual sanity check.
import numpy as np
import matplotlib.pyplot as plt

chunk = np.load(paths[0])
images = chunk["images"]

fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for ax, img in zip(axes.ravel(), images[:32]):
    ax.imshow(img[0], cmap="magma", vmin=0, vmax=1)
    ax.axis("off")
plt.tight_layout()